# Student Feedback Analyzer — Streamlit Dashboard


In [ ]:
# Install the required libraries and download Cloudflare Tunnel.
!pip -q install streamlit pandas scikit-learn matplotlib
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print("Setup complete. Continue to the next cell.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.9 MB/s eta 0:00:00
Setup complete. Continue to the next cell.


In [ ]:
%%writefile /content/app.py
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

st.set_page_config(
    page_title="Student Feedback Analyzer",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded",
)

LABELS = ["Positive", "Negative", "Neutral"]

POSITIVE = [
    "The teachers explain concepts clearly.", "The teacher is helpful and supportive.",
    "I really enjoy attending these classes.", "The course content is excellent.",
    "The learning materials are very useful.", "The classroom environment is comfortable.",
    "I am satisfied with the teaching quality.", "The practical sessions are interesting.",
    "The staff responds quickly to questions.", "The lectures are engaging and informative.",
    "The library facilities are excellent.", "The assignments helped me understand the topic.",
    "The teacher gives clear examples.", "The online resources are easy to use.",
    "The course is well organized.", "The lab equipment works very well.",
    "The feedback from teachers is constructive.", "The timetable is convenient for students.",
    "The study materials are easy to understand.", "The teacher encourages students to ask questions.",
    "The classroom is clean and bright.", "The support team solved my issue quickly.",
    "The project activities improved my skills.", "The lessons are interactive and enjoyable.",
    "The teacher is patient when explaining doubts.", "The course has improved my confidence.",
    "The practical demonstrations are very helpful.", "The college provides good learning facilities.",
    "The instructions for assignments are clear.", "I had a great learning experience."
]
NEGATIVE = [
    "The teacher explains concepts very poorly.", "I dislike attending these classes.",
    "The course content is confusing.", "The teachers are unhelpful.",
    "The learning materials are useless.", "The classroom is dirty and uncomfortable.",
    "I am dissatisfied with the teaching quality.", "The practical sessions are boring.",
    "The staff never responds to questions.", "The lectures are confusing and difficult.",
    "The library facilities are poor.", "The assignments are too difficult to understand.",
    "The teacher gives unclear explanations.", "The online resources are difficult to access.",
    "The course is badly organized.", "The lab equipment does not work.",
    "The feedback from teachers is not helpful.", "The timetable is inconvenient for students.",
    "The study materials are hard to understand.", "The teacher discourages students from asking questions.",
    "The classroom is noisy and poorly maintained.", "The support team did not solve my issue.",
    "The project activities were not useful.", "The lessons are dull and repetitive.",
    "The teacher is impatient when explaining doubts.", "The course has not improved my confidence.",
    "The practical demonstrations are confusing.", "The college facilities need improvement.",
    "The assignment instructions are unclear.", "I had a frustrating learning experience."
]
NEUTRAL = [
    "The lecture starts at ten in the morning.", "The class has thirty students.",
    "The examination is scheduled for Monday.", "The course contains five modules.",
    "The teacher shared the timetable today.", "The semester begins next week.",
    "The classroom has forty chairs.", "The assignment deadline is Friday.",
    "The library opens at nine in the morning.", "The next lecture is about mathematics.",
    "The class duration is one hour.", "The students submitted their forms today.",
    "The course has three assignments.", "The lab session is on Wednesday.",
    "The lecture hall is on the second floor.", "The semester has sixteen weeks.",
    "The meeting starts at two PM.", "The timetable was updated yesterday.",
    "The class has two sections.", "The exam form is available online.",
    "The course includes six chapters.", "The orientation is scheduled for Tuesday.",
    "The teacher uploaded a document today.", "The college has four academic departments.",
    "The project submission date is next month.", "The library has twelve computers.",
    "The workshop lasts for two days.", "The class begins after lunch.",
    "The notice was posted this morning.", "The course code is CS101."
]

@st.cache_resource
def train_model():
    texts = POSITIVE + NEGATIVE + NEUTRAL
    labels = (
        ["Positive"] * len(POSITIVE)
        + ["Negative"] * len(NEGATIVE)
        + ["Neutral"] * len(NEUTRAL)
    )
    X = pd.Series(texts).apply(clean_text)
    y = pd.Series(labels)
    model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    model.fit(X, y)
    return model, X, y

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_evaluation():
    texts = POSITIVE + NEGATIVE + NEUTRAL
    labels = (
        ["Positive"] * len(POSITIVE)
        + ["Negative"] * len(NEGATIVE)
        + ["Neutral"] * len(NEUTRAL)
    )
    X = pd.Series(texts).apply(clean_text)
    y = pd.Series(labels)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    eval_model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    eval_model.fit(X_train, y_train)
    pred = eval_model.predict(X_test)
    return accuracy_score(y_test, pred), y_test, pred

def extract_keywords(series, n=15):
    cleaned = series.astype(str).apply(clean_text)
    try:
        vec = CountVectorizer(stop_words="english", ngram_range=(1, 2), max_features=n)
        matrix = vec.fit_transform(cleaned)
        return pd.DataFrame({
            "Keyword": vec.get_feature_names_out(),
            "Frequency": matrix.sum(axis=0).A1
        }).sort_values("Frequency", ascending=False)
    except ValueError:
        return pd.DataFrame(columns=["Keyword", "Frequency"])

model, _, _ = train_model()
page = st.sidebar.radio(
    "Navigation",
    ["Analyze Feedback", "Model Evaluation", "View Training Data", "About"],
)
st.sidebar.markdown("---")
st.sidebar.caption("Built with Python • NLP • Scikit-learn • Streamlit")
st.sidebar.info("Learning prototype: predictions may be inaccurate on real-world feedback.")

st.title("📊 Student Feedback Analyzer")
st.caption("Analyze student feedback using Natural Language Processing and Machine Learning.")

if page == "Analyze Feedback":
    st.info("Enter feedback manually or upload a CSV file with a column named `feedback`.")
    manual, upload = st.tabs(["✍️ Enter Feedback", "📁 Upload CSV"])
    feedback_items = []
    with manual:
        text_input = st.text_area(
            "Enter one feedback comment per line",
            height=160,
            placeholder="The teachers are helpful.\nThe classroom is uncomfortable.\nThe exam is on Monday."
        )
        if text_input.strip():
            feedback_items = [x.strip() for x in text_input.splitlines() if x.strip()]
    with upload:
        uploaded = st.file_uploader("Upload feedback CSV", type=["csv"])
        if uploaded is not None:
            try:
                upload_df = pd.read_csv(uploaded)
                if "feedback" not in upload_df.columns:
                    st.error("CSV must contain a column named 'feedback'.")
                else:
                    feedback_items = upload_df["feedback"].dropna().astype(str).tolist()
                    st.success(f"Loaded {len(feedback_items)} feedback rows.")
                    st.dataframe(upload_df.head(), use_container_width=True)
            except Exception as exc:
                st.error(f"Could not read the CSV: {exc}")

    if st.button("🔍 Analyze Feedback", type="primary", use_container_width=True):
        items = [str(x).strip() for x in feedback_items if str(x).strip() and clean_text(x)]
        if not items:
            st.warning("Please enter feedback or upload a valid CSV.")
        else:
            cleaned = [clean_text(x) for x in items]
            predictions = model.predict(cleaned)
            probabilities = model.predict_proba(cleaned).max(axis=1)
            st.session_state["results"] = pd.DataFrame({
                "Feedback": items,
                "Predicted Sentiment": predictions,
                "Model Confidence": probabilities,
            })

    results = st.session_state.get("results")
    if results is not None:
        st.markdown("### Feedback Overview")
        total = len(results)
        counts = results["Predicted Sentiment"].value_counts().reindex(LABELS, fill_value=0)
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Feedback", total)
        c2.metric("Positive", int(counts["Positive"]))
        c3.metric("Negative", int(counts["Negative"]))
        c4.metric("Neutral", int(counts["Neutral"]))

        left, right = st.columns([1, 1])
        with left:
            st.markdown("#### Sentiment Distribution")
            fig, ax = plt.subplots(figsize=(5, 4))
            counts[counts > 0].plot(kind="pie", autopct="%1.1f%%", ax=ax, ylabel="")
            fig.tight_layout()
            st.pyplot(fig)
            plt.close(fig)
        with right:
            st.markdown("#### Frequent Keywords")
            keywords = extract_keywords(results["Feedback"])
            if keywords.empty:
                st.write("Not enough meaningful words found.")
            else:
                st.bar_chart(keywords.set_index("Keyword").head(10))

        st.markdown("#### Feedback Results")
        selected = st.multiselect("Filter sentiment", LABELS, default=LABELS)
        filtered = results[results["Predicted Sentiment"].isin(selected)]
        st.dataframe(
            filtered,
            use_container_width=True,
            hide_index=True,
            column_config={
                "Model Confidence": st.column_config.NumberColumn(
                    "Model Confidence", format="%.1%%",
                    help="Model's highest class probability; not a guarantee of correctness."
                )
            }
        )
        st.download_button(
            "⬇️ Download Results as CSV",
            data=results.to_csv(index=False).encode("utf-8"),
            file_name="student_feedback_analysis.csv",
            mime="text/csv",
            use_container_width=True,
        )

elif page == "Model Evaluation":
    st.subheader("Model Evaluation")
    st.write("Metrics below use a held-out test split of the small sample dataset.")
    accuracy, y_test, predictions = get_evaluation()
    st.metric("Held-out test accuracy", f"{accuracy:.1%}")
    st.warning("This score is only an educational baseline. The sample dataset is small and manually written; it does not establish real-world performance.")
    st.text(classification_report(y_test, predictions, labels=LABELS, zero_division=0))
    fig, ax = plt.subplots(figsize=(6, 4))
    matrix = confusion_matrix(y_test, predictions, labels=LABELS)
    disp = __import__("sklearn.metrics", fromlist=["ConfusionMatrixDisplay"]).ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=LABELS
    )
    disp.plot(ax=ax, colorbar=False)
    plt.xticks(rotation=20)
    fig.tight_layout()
    st.pyplot(fig)
    plt.close(fig)

elif page == "View Training Data":
    train_df = pd.DataFrame({
        "Feedback": POSITIVE + NEGATIVE + NEUTRAL,
        "Sentiment": ["Positive"] * len(POSITIVE) + ["Negative"] * len(NEGATIVE) + ["Neutral"] * len(NEUTRAL)
    })
    st.subheader("Sample Training Dataset")
    st.dataframe(train_df, use_container_width=True, hide_index=True)
    st.bar_chart(train_df["Sentiment"].value_counts())

else:
    st.subheader("About this Project")
    st.markdown("""
    **Objective:** Analyze written student feedback and estimate its sentiment.

    **Workflow**
    1. Collect feedback manually or from CSV.
    2. Clean text by normalizing case and removing punctuation.
    3. Convert text to numeric features with TF-IDF.
    4. Predict sentiment using Logistic Regression.
    5. Visualize sentiment counts and frequent keywords.
    6. Export results to CSV.

    **Tech stack:** Python, Pandas, scikit-learn, TF-IDF, Logistic Regression, Matplotlib, Streamlit.

    **Limitations:** The model is trained on a small illustrative dataset and may struggle with sarcasm, mixed sentiment, Hindi/Hinglish, spelling mistakes, or unfamiliar topics. Do not use it for high-stakes decisions.
    """)


Writing /content/app.py


In [ ]:
import subprocess, time, re, threading, sys

# Start Streamlit in the background.
streamlit_log = open("/content/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "/content/app.py",
     "--server.port", "8501", "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=streamlit_log, stderr=subprocess.STDOUT
)

# Start a temporary public Cloudflare tunnel and display its URL.
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

print("Starting Streamlit and creating a temporary public URL...")
print("Keep this Colab cell running while you use the dashboard.")
found_url = False
try:
    for line in tunnel_process.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and not found_url:
            print("\nYOUR DASHBOARD URL:", match.group(0))
            print("Open that URL in a new browser tab.")
            found_url = True
        if not found_url and ("ERR" in line or "error" in line.lower()):
            print(line.rstrip())
except KeyboardInterrupt:
    print("Tunnel cell interrupted. The temporary dashboard may stop.")


Starting Streamlit and creating a temporary public URL...
Keep this Colab cell running while you use the dashboard.

YOUR DASHBOARD URL: https://nat-webcast-entered-tries.trycloudflare.com
Open that URL in a new browser tab.
